# 12.06 · Evaluate, merge, export — Milestone M6

**Chapter:** Fine-tuning & Post-training · **Milestone:** M6

## Goals
- [ ] Merge a LoRA adapter into the base weights
- [ ] Check for **regressions** on general ability (catastrophic forgetting) with `lm-evaluation-harness`
- [ ] Export to Ollama (quantised GGUF) and serve the fine-tuned model locally behind the OpenAI-compatible API
- [ ] Share on the Hugging Face Hub; know model merging (mergekit)

## Resources
- [EleutherAI lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) — free
- [Ollama – Importing a model](https://docs.ollama.com/import) · [llama.cpp – convert & quantize](https://github.com/ggml-org/llama.cpp) — free
- [mergekit](https://github.com/arcee-ai/mergekit) · [HF – Merge LLMs with mergekit](https://huggingface.co/blog/mlabonne/merge-models) — free

**Run 12.03 first** — this notebook uses its adapter (`data/ft/smollm-lora`).

In [ ]:
import json
import os
import subprocess
import time

# transformers 5 loads weights in parallel threads; straight onto MPS (lm_eval's device_map path) that segfaults. Load sequentially.
os.environ.setdefault("HF_DEACTIVATE_ASYNC_LOAD", "1")

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from aieng import DATA
from aieng.llm import client
from task import SYSTEM, dataset, messages, score

FT = DATA / "ft"
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"
ADAPTER, MERGED = FT / "smollm-lora", FT / "smollm-payments-merged"
assert ADAPTER.exists(), "Adapter not found: run 12.03 first."
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

## 1. Merge the adapter
Serving an adapter needs PEFT at inference (or an engine with LoRA support, e.g. vLLM multi-LoRA). Merging bakes $W + \frac{\alpha}{r}BA$ into plain
weights: same speed as the base model, works everywhere.

In [ ]:
tok = AutoTokenizer.from_pretrained(ADAPTER)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
print(sorted(p.name for p in MERGED.iterdir()))
print(f"merged size: {sum(p.stat().st_size for p in MERGED.iterdir()) / 1e6:.0f} MB")

## 2. Did we break anything? General benchmarks before/after
Fine-tuning on a narrow task can erode general skills. Run a couple of quick general benchmarks on base vs merged (small `limit` to stay fast —
use the full sets for real decisions).

In [ ]:
import lm_eval

def bench(path, tasks=("arc_easy", "hellaswag"), limit=200):
    out = lm_eval.simple_evaluate(model="hf", model_args=f"pretrained={path},dtype=float32", tasks=list(tasks), limit=limit, device=device,
                                  batch_size=16, log_samples=False, verbosity="ERROR")
    return {t: round(out["results"][t].get("acc_norm,none", out["results"][t].get("acc,none")), 3) for t in tasks}


start = time.perf_counter()
print("base  :", bench(BASE))
print("merged:", bench(str(MERGED)))
print(f"({time.perf_counter() - start:.0f}s)")

Small differences at `limit=200` are noise (±3 points). A big drop means you overtrained: lower LR/epochs, smaller rank, or mix in general
instruction data. (Also check refusals/safety behaviour if the base model had them.)

## 3. Export to Ollama (quantised)
Ollama imports safetensors directly and can quantise on import. The `Modelfile` sets the prompt template, system prompt and sampling defaults. **Always set `TEMPLATE`** to the model's chat format and
check it with `ollama show <name> --template`: a missing template is the #1 reason an imported fine-tune looks broken.

In [ ]:
# Ollama does not read HF's chat_template.jinja: without TEMPLATE it sends the raw prompt (no system, no ChatML) and the model babbles.
CHATML = """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""
modelfile = MERGED / "Modelfile"
modelfile.write_text(f'FROM {MERGED}\nTEMPLATE """{CHATML}"""\nSYSTEM """{SYSTEM}"""\n'
                     'PARAMETER stop <|im_end|>\nPARAMETER temperature 0\nPARAMETER num_predict 64\n')
name = "smollm-payments"
r = subprocess.run(["ollama", "create", name, "-f", str(modelfile), "--quantize", "int8"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-500:]
print(subprocess.run(["ollama", "show", name], capture_output=True, text=True).stdout)

`int8` keeps near-full quality; `int4` is the usual size/quality trade for bigger models (tiny models lose more from 4-bit — measure it).
Ollama ≥0.40 names these `int4`/`int8` (older versions and llama.cpp use GGUF names like `q8_0`, `q4_K_M`).
The llama.cpp route does the same by hand: `convert_hf_to_gguf.py` → `llama-quantize` → `llama-server` (13.01).

## 4. Serve and evaluate the exported model through the OpenAI-compatible API
The model now behaves like any other provider in `aieng.llm` — every earlier lesson's code works with it.

In [ ]:
oc, _ = client("ollama")
test_ood = [e for e in dataset(800, seed=3) if e["text"].startswith(("Netflix charged me", "me cobraron"))][:50]
start = time.perf_counter()
scores = []
for e in test_ood:
    out = oc.chat.completions.create(model=name, messages=[{"role": "user", "content": e["text"]}], temperature=0, max_tokens=64).choices[0].message.content
    scores.append(score(out, e["target"]))
secs = (time.perf_counter() - start) / len(test_ood)
print(f"{name} via Ollama: exact {sum(s['exact'] for s in scores) / len(scores):.0%} on held-out templates | {secs * 1000:.0f} ms/request")
print(oc.chat.completions.create(model=name, messages=[{"role": "user", "content": "pagué 45 soles el 3 de mayo de 2026 por el gas"}]).choices[0].message.content)

## 5. Share it (Hugging Face Hub)
Runs if `HF_TOKEN` is set. Write a model card: base model, data, task, eval results, limitations, license (inherits the base model's: Apache-2.0 here).
```python
merged.push_to_hub("your-username/smollm2-135m-payments", private=True); tok.push_to_hub("your-username/smollm2-135m-payments", private=True)
```

## 6. Model merging (beyond adapters)
mergekit combines *different fine-tunes of the same base* without training: linear/SLERP averaging, TIES/DARE (resolve conflicting weight deltas),
task arithmetic (add/subtract "task vectors"). Cheap way to combine skills; evaluate the result like any model.
```yaml
# mergekit config: average two LoRA-merged fine-tunes of the same base
merge_method: linear
models: [{model: ./payments-merged, parameters: {weight: 0.5}}, {model: ./other-task-merged, parameters: {weight: 0.5}}]
dtype: float16
```

## Milestone M6 — deliverable
- [ ] A fine-tuned small model that **beats its base** (and ideally matches a bigger model) on your eval, including held-out cases
- [ ] General-benchmark check showing no major regression
- [ ] Exported (GGUF/Ollama) and served locally through an OpenAI-compatible API
- [ ] A model card with data, method, results and limitations

## Exercises
1. Quantise to `int4` too and compare exact-match and speed with `int8`. How much does a 135M model suffer from 4-bit?
2. Fine-tune on **your own** task (from your M3/M4 project) end to end: data → LoRA → eval → export → serve.
3. Fuse the MLX adapter from 12.03 (`python -m mlx_lm fuse ...`) and import it into Ollama the same way.

In [ ]:
# your code here